In [2]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Times New Roman'
from scipy.stats import norm
import import_ipynb
from monte_carlo_failure import (
    monte_carlo_failure,
    Pf_to_beta,
    beta_to_Pf,
)


def compute_d_grown(D_base, alpha_traffic, T_max):
    """
    This computes damage through each year, taking into account exponential traffic growth
    returns annual_dammage as an array, each item represents damage in that time
    """
    if alpha_traffic == 0.0:
        return D_base * np.arange(1, T_max + 1, dtype=float)
    k = np.arange(1, T_max + 1)
    annual_damage = D_base * (1.0 + alpha_traffic) ** k
    return np.cumsum(annual_damage)


In [3]:
def calibration_sweep(
    D_base,
    alpha_traffic,
    beta_targets=(1.0, 3.0),
    T_design=75,
    x_F_grid=None,
    N_samples=500_000,
    delta_mean=1.0, delta_cov=0.30,
    e_mean=1.0, e_cov=0.15,
    seed=None,
    print_statement=True,
):
    """
    For each beta_target, solve for the x_F that yields beta(T_design) = beta_target.

    returns:
    calibrated, a dict { beta_target: x_F_calibrated }
    sweep, a dict { beta_target: (x_F_grid, beta_at_T, Pf_at_T) }
    """
    if x_F_grid is None:
        x_F_grid = np.linspace(0.05, 4.0, 50)

    T_max = T_design + 5
    d_grown = compute_d_grown(D_base, alpha_traffic, T_max)
    t_values = np.array([T_design], dtype=float)

    calibrated = {}
    sweep      = {}

    for beta_target in beta_targets:
        Pf_at_T   = np.zeros_like(x_F_grid, dtype=float)
        beta_at_T = np.zeros_like(x_F_grid, dtype=float)

        for j, x_F in enumerate(x_F_grid):
            Pf = monte_carlo_failure(
                t_values=t_values, d_grown_array=d_grown, x_F=x_F,
                N_samples=N_samples,
                delta_mean=delta_mean, delta_cov=delta_cov,
                e_mean=e_mean,         e_cov=e_cov,
                seed=seed,
            )
            Pf_at_T[j]   = float(Pf[0])
            beta_at_T[j] = float(Pf_to_beta(Pf[0]))

        # interpolate inverse
        valid = np.isfinite(beta_at_T)
        if not valid.any():
            x_F_cal = float("nan")
        else:
            x_F_cal = float(
                np.interp(beta_target,
                          beta_at_T[valid][::-1],
                          x_F_grid[valid][::-1])
            )

        calibrated[beta_target] = x_F_cal
        sweep[beta_target]      = (x_F_grid, beta_at_T, Pf_at_T)

        if print_statement:
            print(f"  β_target = {beta_target}   would require  x_F = {x_F_cal:.4f}")

    return calibrated, sweep


def plot_calibration(calibrated, sweep, T_design=75, alpha_traffic=0.025, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 5))

    palette = {1.0: "#4477AA", 3.0: "#CC6677"}
    for beta_target, (x_F_grid, beta_at_T, _) in sweep.items():
        valid = np.isfinite(beta_at_T)
        color = palette.get(beta_target, None)
        ax.plot(x_F_grid[valid], beta_at_T[valid], marker='o',
                linewidth=2, color=color, label=f'β at T = {T_design} yr')
        ax.axhline(beta_target, linestyle='--', color=color, linewidth=1, alpha=0.7,
                   label=f'Target β = {beta_target}')
        ax.axvline(calibrated[beta_target], linestyle=':', color=color, linewidth=1, alpha=0.7,
                   label=f'Calibrated x_F = {calibrated[beta_target]:.3f}')

    ax.set_xlabel('Load factor $x_F$')
    ax.set_ylabel(f'Reliability index β at t = {T_design} yr')
    ax.set_title(f'Calibration of $x_F$ (α = {alpha_traffic*100:.1f}%/yr)')
    ax.legend(fontsize=9, ncol=2)
    ax.grid(True, alpha=0.4)
    ax.set_xlim(left=0.5)
    return ax


In [4]:
def sensitivity_sweep(
    D_base,
    x_F,
    alpha_scenarios=(0.005, 0.01, 0.02, 0.025, 0.03, 0.04, 0.05),
    T_design=75,
    T_horizon=None,
    N_samples=500_000,
    delta_mean=1.0, delta_cov=0.30,
    e_mean=1.0, e_cov=0.15,
    seed=None,
    plot=True,
    beta_reference_lines=(1.0, 3.0),
):
    """
    This assumes a fixed x_F, and then sweeps alpha_traffic over a set of growth scenarios and
    returns β(t) and P_f(t) curves. Optionally, it can plot
    """
    if T_horizon is None:
        T_horizon = T_design + 25
    years = np.arange(1, T_horizon + 1, dtype=float)

    curves = {}
    for alpha in alpha_scenarios:
        d_grown = compute_d_grown(D_base, alpha, T_horizon)
        Pf = monte_carlo_failure(
            t_values=years, d_grown_array=d_grown, x_F=x_F,
            N_samples=N_samples,
            delta_mean=delta_mean, delta_cov=delta_cov,
            e_mean=e_mean,         e_cov=e_cov,
            seed=seed,
        )
        curves[alpha] = {
            "years": years,
            "Pf":    Pf,
            "beta":  Pf_to_beta(Pf),
        }

    if plot:
        _plot_sensitivity(curves, x_F, T_design, beta_reference_lines)

    return curves


def _plot_sensitivity(curves, x_F, T_design, beta_reference_lines):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cmap = plt.get_cmap("viridis")
    n = len(curves)

    for i, (alpha, c) in enumerate(sorted(curves.items())):
        color = cmap(i / max(1, n - 1))
        axes[0].plot(c["years"], c["Pf"],   color=color, linewidth=2,
                     label=f'α = {alpha*100:.1f}%')
        axes[1].plot(c["years"], c["beta"], color=color, linewidth=2,
                     label=f'α = {alpha*100:.1f}%')

    axes[0].axvline(T_design, linestyle="--", color="black", linewidth=1)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Time (years)")
    axes[0].set_ylabel("$P_f(t)$")
    axes[0].set_title("$P_f(t)$ sensitivity to growth rate")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.4)
    axes[0].set_ylim(bottom=5e-06)

    axes[1].axvline(T_design, linestyle="--", color="black", linewidth=1)
    for bt in beta_reference_lines:
        axes[1].axhline(bt, linestyle=":", color="red", linewidth=1, alpha=0.6)
    axes[1].set_xlabel("Time (years)")
    axes[1].set_ylabel("Reliability index β(t)")
    axes[1].set_title("β(t) sensitivity to growth rate")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.4)
    axes[1].set_ylim(top=4)

    plt.suptitle(f"Growth-rate sensitivity  |  $x_F$ = {x_F:.3f} fixed", y=1.01)
    plt.tight_layout()
    plt.show()
